In [ ]:
!pip install findspark

import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("count_countDistinct_approx_count_countDistinct_").getOrCreate()

In [ ]:
df = spark.read.parquet("./data/dataframe.parquet")

In [ ]:
df.printSchema()

root
 |-- nombre: string (nullable = true)
 |-- color: string (nullable = true)
 |-- cantidad: long (nullable = true)



In [ ]:
df.show()

+------+-----+--------+
|nombre|color|cantidad|
+------+-----+--------+
|  Jose| azul|    1900|
|  NULL| NULL|    1700|
|  NULL| rojo|    1300|
|  Juan| rojo|    1500|
+------+-----+--------+



In [ ]:
# Count es importante renombrar las columnas para ser mas informativas (no cuenta los NULL)

from pyspark.sql.functions import count

df.select(count("nombre").alias("conteo nombre"), count("color").alias("conteo color")).show()

+-------------+------------+
|conteo nombre|conteo color|
+-------------+------------+
|            2|           3|
+-------------+------------+



In [ ]:
# como podemos hacer para contarlos con todos los registros (no cuenta los NULL)

from pyspark.sql.functions import count
df.select(count("nombre").alias("conteo nombre"), count("color").alias("conteo color"), count("*").alias("total registros")).show()

+-------------+------------+---------------+
|conteo nombre|conteo color|total registros|
+-------------+------------+---------------+
|            2|           3|              4|
+-------------+------------+---------------+



## Funcion countDistinct()

In [ ]:
#countDistinct

from pyspark.sql.functions import countDistinct

df.select(
    countDistinct("color").alias("colores diferentes")
).show()

+------------------+
|colores diferentes|
+------------------+
|                 2|
+------------------+



## approx_count_distinct()

In [ ]:
dataframe = spark.read.parquet("./data/vuelos/vuelos.parquet")

hay que tener en cuenta que contar el numero exacto de elementos unicos en cada grupo en un gran conjunto de datos es una operacion costosa y que se requiere mucho tiempo. En algunos casos es suficiente a veces tener un recuento unico aproximado y precisamente es lo que nos va a proporcionar esta funcion. Nos dara el conteo aproximado que tiene una columna

In [ ]:
dataframe.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- FLIGHT_NUMBER: integer (nullable = true)
 |-- TAIL_NUMBER: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DESTINATION_AIRPORT: string (nullable = true)
 |-- SCHEDULED_DEPARTURE: integer (nullable = true)
 |-- DEPARTURE_TIME: integer (nullable = true)
 |-- DEPARTURE_DELAY: integer (nullable = true)
 |-- TAXI_OUT: integer (nullable = true)
 |-- WHEELS_OFF: integer (nullable = true)
 |-- SCHEDULED_TIME: integer (nullable = true)
 |-- ELAPSED_TIME: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- WHEELS_ON: integer (nullable = true)
 |-- TAXI_IN: integer (nullable = true)
 |-- SCHEDULED_ARRIVAL: integer (nullable = true)
 |-- ARRIVAL_TIME: integer (nullable = true)
 |-- ARRIVAL_DELAY: integer (null

In [ ]:
# queremos el numero de aerolineas que tiene esta base de datos (nos dio un valoe aprox al valor real)
from pyspark.sql.functions import approx_count_distinct

dataframe.select(
    countDistinct("AIRLINE"),
    approx_count_distinct("AIRLINE")
).show()

+-----------------------+------------------------------+
|count(DISTINCT AIRLINE)|approx_count_distinct(AIRLINE)|
+-----------------------+------------------------------+
|                     14|                            13|
+-----------------------+------------------------------+

